## Import

In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np
from functools import reduce
from datetime import datetime
import re

In [2]:
def replace_outliers_with_nan(df, sd_thresh=3, exclude_cols=None):
    """
    Iterates through ALL columns in the dataframe.
    Replaces values exceeding +/- sd_thresh with NaN.
    
    Parameters:
    - exclude_cols: List of column names to skip (e.g., ['SubjectID', 'Block'])
    """
    # Handle default mutable argument
    if exclude_cols is None:
        exclude_cols = []
        
    for column in df.columns:
        # 1. Skip excluded columns explicitly
        if column in exclude_cols:
            continue

        # 2. Skip non-numeric columns (like string types)
        if not pd.api.types.is_numeric_dtype(df[column]):
            continue

        # 3. Calculate Stats
        mean = df[column].mean()
        std = df[column].std()
        
        upper = mean + (sd_thresh * std)
        lower = mean - (sd_thresh * std)

        # 4. Vectorized Mask (True = Outlier)
        outlier_mask = (df[column] > upper) | (df[column] < lower)
        
        # 5. Logging
        n_outliers = outlier_mask.sum()
        if n_outliers > 0:
            print(f"[Log] {column}: Replaced {n_outliers} outliers ({n_outliers/len(df):.2%} of data).")
        
        # 6. In-place Replacement
        df.loc[outlier_mask, column] = np.nan
        
    return df

def replace_outliers_with_nan_cols(df, columns_to_check, sd_thresh=3):
    """
    Replaces outliers with NaN.
    Raises KeyError if a column is missing.
    Returns the modified dataframe if successful.
    """
    # 1. Validation: Check all columns exist first
    # If this fails, the error is raised and nothing is returned.
    missing_cols = [col for col in columns_to_check if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns in dataframe: {missing_cols}")

    # 2. Process columns
    for column in columns_to_check:
        if not pd.api.types.is_numeric_dtype(df[column]):
            print(f"Skipping non-numeric column: {column}")
            continue

        mean = df[column].mean()
        std = df[column].std()
        
        upper = mean + (sd_thresh * std)
        lower = mean - (sd_thresh * std)

        # Vectorized replacement
        outlier_mask = (df[column] > upper) | (df[column] < lower)
        
        # Logging
        n_outliers = outlier_mask.sum()
        if n_outliers > 0:
            print(f"[Log] {column}: Replaced {n_outliers} outliers ({n_outliers/len(df):.2%} of data).")
        else:
            print(f"[Log] {column}: No outliers found.")
        
        # In-place Replacement
        df.loc[outlier_mask, column] = np.nan

    return df


def find_newest_file(path): 
    matching_files = glob(path)
    
    # Check if any files were found
    if not matching_files:
        print("No matching files found.")
    else:
        # Find the newest file based on modification time
        new_file_path = max(matching_files, key=os.path.getmtime)
        # df = pd.read_csv(new_file_path)
        # print(f"Found {len(matching_files)} matching files.")
        print(f"The newest file is: {new_file_path}")

        return new_file_path

In [3]:
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"

session = "s1_r1"
# id_matrix = pd.Series(list(range(3000000, 3000401)), name = "sub")
ra_tracker_path = f"/home/data/NDClab/analyses/thrive-theta-ddm/code/preprocessing-eeg/ra_tracker_labeled_flanker_{session}.csv"
checked_path = f"{dataset_path}/sourcedata/checked/"
derivatives_path = f"{dataset_path}/derivatives/"

ra_tracker = pd.read_csv(ra_tracker_path)

if session == "s1_r1":
    session_csv = "s1r1"
    exclude_id_list = [3000242, 3000254, 3000255] # these subs are excluded from ALL behavior + ALL EEG
    unusable_soc = [3000047, 3000056, 3000180, 3000182] # these subs are excluded from SOC behavior + SOC EEG + SOC surveys
    eeg_unusable_nonsoc = [3000008] # these subs are excluded from NONSOC EEG
    exclude_eeg = [3000080, 3000161, 3000228] # these subs are excluded from ALL EEG
    exclude_initstated = [3000066]
    exclude_subset_dyadb = [3000361]

iqs_p_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thriveiqsparent*{session_csv}*.csv")
iqs_ch_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thriveiqschild*{session_csv}*.csv")
bbs_p_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thrivebbsparent*{session_csv}*.csv")
bbs_ch_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/Thrivebbschild*{session_csv}*.csv")
bbs_ra_path = find_newest_file(f"{derivatives_path}/preprocessed/redcap/ThrivebbsRA*{session_csv}*.csv")

behavivor_summary_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/summary*{session}*.csv")
behav_trial_data_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/full_df*.csv")

ern_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_erp_2*.csv")
ern_laplacian_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_erp_laplacian_2*.csv")
tf_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_power_itps*.csv")
icps_path = find_newest_file(f"{analysis_path}/derivatives/csv/{session}/thrive_icps*.csv")

ddm_path = find_newest_file(f"{analysis_path}/derivatives/behavior/{session}/ddm_fit_{session}*.csv")
# total 16 subjects flagged; out of them: 4 are fine after inspection, 1 has no EEG data (properly described in txt); total 11 subjects needed to be flagged (see lists above)
# ra_tracker

The newest file is: /home/data/NDClab/datasets/thrive-dataset//derivatives//preprocessed/redcap/Thriveiqsparents1r1_SCRD_2025-12-22_1526.csv
The newest file is: /home/data/NDClab/datasets/thrive-dataset//derivatives//preprocessed/redcap/Thriveiqschilds1r1_SCRD_2025-12-22_1526.csv
The newest file is: /home/data/NDClab/datasets/thrive-dataset//derivatives//preprocessed/redcap/Thrivebbsparents1r1_SCRD_2025-12-22_1526.csv
The newest file is: /home/data/NDClab/datasets/thrive-dataset//derivatives//preprocessed/redcap/Thrivebbschilds1r1_SCRD_2025-12-22_1525.csv
The newest file is: /home/data/NDClab/datasets/thrive-dataset//derivatives//preprocessed/redcap/ThrivebbsRAs1r1_SCRD_2025-12-22_1526.csv
The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/behavior/s1_r1/summary_s1_r1_15_01_2026_16_55_11.csv
The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/behavior/s1_r1/full_df_15_01_2026_16_55_11.csv
The newest file is: /home/data/NDClab/analyses/

## Age

In [4]:
# 1. Load and Filter Columns safely
age_data = pd.read_csv(iqs_p_path)

# Identify columns explicitly to avoid index slicing errors
age_cols = [c for c in age_data.columns if "agemos" in c]

# Select only necessary columns
age_data = age_data[["record_id"] + age_cols].copy()

# 2. Rename and Transform ID
age_data = age_data.rename(columns={"record_id": "sub"})
age_data["sub"] = age_data["sub"] - 80000

# 3. Coalesce Age Columns (The "English/Spanish" merge)
# bfill(axis=1) fills NaNs with the next valid value in the row. 
# We then take the first column, effectively grabbing the first non-null value found.
age_data["age_m"] = age_data[age_cols].bfill(axis=1).iloc[:, 0]

# 4. Drop rows where we couldn't find ANY age
age_data = age_data.dropna(subset=["age_m"]).reset_index(drop=True)

# 5. Filter final columns
age_data = age_data[["sub", "age_m"]]

# 6. Apply Session Adjustments
if session == "s2_r1":
    age_data["age_m"] += 9
elif session == "s3_r1":
    age_data["age_m"] += 18

## Sex

In [5]:
# 1. Load Data
sex_data = pd.read_csv(iqs_p_path)

# 2. Rename ID safely
sex_data = sex_data.rename(columns={"record_id": "sub"})
sex_data["sub"] = sex_data["sub"] - 80000

# 3. Define columns
sex_cols = [c for c in sex_data.columns if f"sexbirth_{session}_e1" in c]

# 4. Vectorized Merge (No loops)
# "Take values from English column. If null, fill with Spanish column."
sex_data["sex"] = sex_data[sex_cols].bfill(axis=1).iloc[:, 0]

# 5. Drop rows where sex is still NaN
sex_data = sex_data.dropna(subset=["sex"]).reset_index(drop=True)

# 6. Ensure Integer Type (Safety Step)
# This handles the float '1.0' issue common with NaNs
sex_data["sex"] = sex_data["sex"].astype(int)

# 7. Final Selection
sex_data = sex_data[["sub", "sex"]]

## Order of flanker (first observed vs first alone)

In [6]:
# 1. Load Data
full_behavior = pd.read_csv(behav_trial_data_path)

# 2. Extract Condition from the First Trial
first_soc = full_behavior.loc[full_behavior["trial_num"] == 1, ["sub", "condition_soc"]].copy(deep=True)

# 3. Rename for clarity
first_soc = first_soc.rename(columns={"condition_soc": "first_soc"})

# Ensure every subject has exactly one entry for "trial 1". 
# Duplicates imply data errors (e.g., merged files); Missing implies data loss.
if first_soc["sub"].duplicated().any():
    print("Warning: Duplicate trial_num=1 found for some subjects. Check data merging.")
    
first_soc = first_soc.reset_index(drop=True)

## DP mode (in-person / online)

In [7]:
# 1. Load Data
bbs_ra = pd.read_csv(bbs_ra_path)

# 2. Define Columns dynamically
acid_col = f"bbsratrk_acid_{session}_e1"
dpid_col = f"bbsratrk_dpid_{session}_e1"

# 3. Filter for in-person kids (Subject ID check)
# Safety: Ensure we are comparing numbers to numbers
bbs_ra = bbs_ra[bbs_ra[acid_col] >= 3000000].reset_index(drop=True)

# 4. Rename and Deduplicate
bbs_ra = bbs_ra.rename(columns={acid_col: "sub"})
bbs_ra = bbs_ra.drop_duplicates(subset=['sub'], keep='last').reset_index(drop=True)

# 5. Determine Partner Mode (Vectorized & Type-Safe)
# Convert to string first to safely check prefixes
dpid_str = bbs_ra[dpid_col].astype(str)

conditions = [
    dpid_str.str.startswith("300"), # Condition 1: In-person (starts with 300)
    dpid_str.str.startswith("10")   # Condition 2: Remote (starts with 10)
]
choices = [1, 0] # 1 = In-person, 0 = Remote

# np.select is much faster and cleaner than a list comprehension with nested ifs
bbs_ra["dp_inperson"] = np.select(conditions, choices, default=np.nan)

# 6. Final Selection
dp_mode = bbs_ra[["sub", "dp_inperson"]]

## EEG

### Subset valid data

In [8]:
# here all valid IDs for EEG are created, which is based on behavioral data
behavior_df = pd.read_csv(behavivor_summary_path)

# now, to perform condition-wise outlier removal, we need to subset nonsocial and social conditions and perform removal separately

# subset non-social valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

valid_behavior_nonsoc = behavior_df_nonsoc["sub"].to_frame()

# subset social valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]
behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

valid_behavior_soc = behavior_df_soc["sub"].to_frame()

Full nonsoc-DF length: 240
Removing subjects [3000242, 3000254, 3000255] from nonsocial condition data
New nonsoc-DF length: 239 

[Log] invalid_rt_percent_nonsoc: Replaced 9 outliers (3.77% of data).
[Log] skipped_percent_nonsoc: Replaced 6 outliers (2.51% of data).
Final non-soc-DF length: 224 

Full soc-DF length: 247
Removing subjects [3000047, 3000056, 3000180, 3000182] from social condition data
Removing subjects [3000242, 3000254, 3000255] from social condition data
New soc-DF length: 241 

[Log] invalid_rt_percent_soc: Replaced 6 outliers (2.49% of data).
[Log] skipped_percent_soc: Replaced 9 outliers (3.73% of data).
Final soc-DF length: 227 



In [9]:
# all IDs from the previous step to merge with all available EEG
id_matrix = valid_behavior_nonsoc.merge(valid_behavior_soc, on="sub", how="outer")

ern_data = pd.read_csv(ern_path)
ern_laplacian_data = pd.read_csv(ern_laplacian_path)
tf_data = pd.read_csv(tf_path)
icps_data = pd.read_csv(icps_path)

data_frames = [
    id_matrix,
    ern_data,
    ern_laplacian_data,
    tf_data,
    icps_data,
]

# the merge is left relative to behav_id
# here all EEG data (ERP + power + ITPS + ICPS) is created
eeg_data = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)

print(f"Full EEG DF length: {eeg_data.shape[0]}")
print(f"Removing subjects with unusable EEG: {exclude_eeg}")
eeg_data = eeg_data[~eeg_data["sub"].isin(exclude_eeg)].reset_index(drop=True)
print(f"Removing subjects with unusable behavior: {exclude_id_list}")
eeg_data = eeg_data[~eeg_data["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New EEG DF length: {eeg_data.shape[0]} \n")

# EEG data is divided to soc and nonsoc
eeg_data_soc = eeg_data[[i for i in eeg_data.columns if ("_soc" in i or i == "sub")]]
eeg_data_soc = eeg_data_soc[eeg_data_soc["sub"].isin(valid_behavior_soc["sub"])].reset_index(drop=True)
print(f"Full SOC EEG DF length: {eeg_data_soc.shape[0]}")
print(f"Removing subjects with unusable SOC data: {unusable_soc}")
eeg_data_soc = eeg_data_soc[~eeg_data_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"New SOC EEG DF length: {eeg_data_soc.shape[0]} \n")

eeg_data_nonsoc = eeg_data[[i for i in eeg_data.columns if ("_nonsoc" in i or i == "sub")]]
eeg_data_nonsoc = eeg_data_nonsoc[eeg_data_nonsoc["sub"].isin(valid_behavior_nonsoc["sub"])].reset_index(drop=True)
print(f"Full NONSOC EEG DF length: {eeg_data_nonsoc.shape[0]}")
print(f"Removing subjects with unusable NONSOC EEG data: {eeg_unusable_nonsoc}")
eeg_data_nonsoc = eeg_data_nonsoc[~eeg_data_nonsoc["sub"].isin(eeg_unusable_nonsoc)].reset_index(drop=True)
print(f"New NONSOC EEG DF length: {eeg_data_nonsoc.shape[0]} \n")

Full EEG DF length: 240
Removing subjects with unusable EEG: [3000080, 3000161, 3000228]
Removing subjects with unusable behavior: [3000242, 3000254, 3000255]
New EEG DF length: 237 

Full SOC EEG DF length: 224
Removing subjects with unusable SOC data: [3000047, 3000056, 3000180, 3000182]
New SOC EEG DF length: 224 

Full NONSOC EEG DF length: 221
Removing subjects with unusable NONSOC EEG data: [3000008]
New NONSOC EEG DF length: 221 



In [10]:
# the code below merges valid data (based on behavior) with EEG, separately soc and non based on valid IDs
valid_eeg_soc = valid_behavior_soc.merge(eeg_data_soc, on="sub", how="inner")

print(f"New SOC EEG DF length: {valid_eeg_soc.shape[0]} \n")

valid_eeg_nonsoc = valid_behavior_nonsoc.merge(eeg_data_nonsoc, on="sub", how="inner")
print(f"New NONSOC EEG DF length: {valid_eeg_nonsoc.shape[0]} \n")

# this code merges SOC and NONSOC EEG so that resulting DF has EEG data for both conditions and is ready for list-wise outlier removal
merged_valid_eeg_data = valid_eeg_soc.merge(valid_eeg_nonsoc, on="sub", how="outer")

New SOC EEG DF length: 224 

New NONSOC EEG DF length: 221 



### Difference scores (Error - Correct)

In [11]:
# the code below renames columns to put err and corr in the end of the column name and then creates pairs out of these columns for subsequent difference score computation

tf_columns_original = [i for i in merged_valid_eeg_data.columns if not (
    # exclude ERN/CRN
    "RN_" in i or \
    # exclude sub ID column
    i=="sub"
)]
tf_columns_renamed = ["_".join(c.split("_err_")) + "_err" if "_err_" in c else "_".join(c.split("_corr_")) + "_corr" if "_corr_" in c else np.nan for c in tf_columns_original]

for i, orig_c in enumerate(tf_columns_original):
    merged_valid_eeg_data.rename({orig_c: tf_columns_renamed[i]}, axis=1, inplace=True)

columns = tf_columns_renamed

# Function to isolate pairs
def isolate_pairs(columns):
    pairs = []
    seen = set()

    for col in columns:
        parts = col.split('_')
        measure, condition, window, accuracy = parts[0], parts[1], parts[2], parts[-1]

        # Create a base identifier without the accuracy part
        base_id = '_'.join(parts[:-1])

        if base_id in seen:
            continue

        # Find the corresponding pair
        if accuracy == 'err':
            corr_col = f"{base_id}_corr"
        else:
            corr_col = f"{base_id}_err"

        if corr_col in columns:
            pairs.append((col, corr_col))
            seen.add(base_id)

    return pairs

# Get the pairs
pairs = isolate_pairs(columns)

# Print the pairs
# for pair in pairs:
#     print(pair)

In [12]:
# before computing difference score, outlier removal must be done on the EEG measures (this dataset should only contain EEG)
merged_valid_eeg_data = replace_outliers_with_nan(merged_valid_eeg_data, sd_thresh=3, exclude_cols=["sub"])

[Log] ERN_soc_laplacian: Replaced 1 outliers (0.42% of data).
[Log] CRN_soc_laplacian: Replaced 2 outliers (0.84% of data).
[Log] power_soc_early_err: Replaced 2 outliers (0.84% of data).
[Log] power_soc_late_err: Replaced 2 outliers (0.84% of data).
[Log] power_soc_early_corr: Replaced 1 outliers (0.42% of data).
[Log] ITPS_soc_early_err: Replaced 1 outliers (0.42% of data).
[Log] ITPS_soc_late_err: Replaced 6 outliers (2.53% of data).
[Log] ITPS_soc_early_corr: Replaced 3 outliers (1.27% of data).
[Log] ITPS_soc_late_corr: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_early_DLPFC_L_err: Replaced 3 outliers (1.27% of data).
[Log] ICPS_soc_early_DLPFC_R_err: Replaced 3 outliers (1.27% of data).
[Log] ICPS_soc_early_OCC_L_err: Replaced 3 outliers (1.27% of data).
[Log] ICPS_soc_early_OCC_R_err: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_early_MOTOR_L_err: Replaced 1 outliers (0.42% of data).
[Log] ICPS_soc_early_MOTOR_R_err: Replaced 1 outliers (0.42% of data).
[Log] ICPS

In [13]:
# computation of difference score for all TF measures

# Iterate only through the column pairs
for p in pairs:
    # 1. Define the new column name dynamically
    # Example: 'tf_soc' -> 'tf_soc_diff'
    diff_col_name = "_".join(p[0].split("_")[:-1]) + "_diff"
    
    # 2. Vectorized Subtraction
    # Pandas automatically handles NaNs: 
    # If p[0] is NaN OR p[1] is NaN, the result is automatically NaN.
    merged_valid_eeg_data[diff_col_name] = (
        merged_valid_eeg_data[p[0]] - merged_valid_eeg_data[p[1]]
    )

In [14]:
# computation of difference score for all ERP
suffixes = ['soc', 'nonsoc', 'soc_laplacian', 'nonsoc_laplacian']

for suffix in suffixes:
    # specific column names
    ern_col = f'ERN_{suffix}'
    crn_col = f'CRN_{suffix}'
    diff_col = f'ERN_min_CRN_{suffix}'
    
    # Vectorized subtraction
    merged_valid_eeg_data[diff_col] = (
        merged_valid_eeg_data[ern_col] - merged_valid_eeg_data[crn_col]
    )

In [15]:
# outlier removal for all difference scores
merged_valid_eeg_data = replace_outliers_with_nan_cols(merged_valid_eeg_data,
    columns_to_check = [c for c in merged_valid_eeg_data if ("diff" in c or "ERN_min_CRN" in c)])

[Log] power_soc_early_diff: Replaced 1 outliers (0.42% of data).
[Log] power_soc_late_diff: Replaced 4 outliers (1.69% of data).
[Log] ITPS_soc_early_diff: Replaced 2 outliers (0.84% of data).
[Log] ITPS_soc_late_diff: Replaced 1 outliers (0.42% of data).
[Log] ICPS_soc_early_DLPFC_L_diff: Replaced 1 outliers (0.42% of data).
[Log] ICPS_soc_early_DLPFC_R_diff: Replaced 3 outliers (1.27% of data).
[Log] ICPS_soc_early_OCC_L_diff: No outliers found.
[Log] ICPS_soc_early_OCC_R_diff: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_early_MOTOR_L_diff: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_early_MOTOR_R_diff: No outliers found.
[Log] ICPS_soc_late_DLPFC_L_diff: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_late_DLPFC_R_diff: Replaced 2 outliers (0.84% of data).
[Log] ICPS_soc_late_OCC_L_diff: No outliers found.
[Log] ICPS_soc_late_OCC_R_diff: Replaced 1 outliers (0.42% of data).
[Log] ICPS_soc_late_MOTOR_L_diff: Replaced 4 outliers (1.69% of data).
[Log] ICPS_soc_late

### Collapsed scores ((L + R) / 2)

In [16]:
# computation of collapsed (L+R)/2 scores for ICPS measures
# note that there is no outlier removal for those measures
# Identify all 'Left' columns to drive the loop
left_cols = [c for c in merged_valid_eeg_data.columns if '_L_' in c]

for l_col in left_cols:
    # Construct the corresponding 'Right' column name
    r_col = l_col.replace('_L_', '_R_')
    
    # Check if the Right pair exists
    if r_col in merged_valid_eeg_data.columns:
        # Create new name: 
        # 1. Replace '_L_' with '_' (ICPS_soc_early_DLPFC_L_err -> ICPS_soc_early_DLPFC_err)
        # 2. Append '_collapsed'
        new_col = l_col.replace('_L_', '_') + '_collapsed'
        
        # Calculate mean (NaN + Value = NaN)
        merged_valid_eeg_data[new_col] = (merged_valid_eeg_data[l_col] + merged_valid_eeg_data[r_col]) / 2

## Behavior

In [17]:
# this code creates behavioral data to analyze strictly flanker behavioral measures (hence it doesn't care about unusable EEG-only subjects)
behavior_df = pd.read_csv(behavivor_summary_path)

# subset nonsocial valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]

behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["6_or_more_err_nonsoc"] == 1]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# criteria-based removals
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# subset only data-related columns
behavior_df_nonsoc[[i for i in behavior_df_nonsoc if ("sub" in i or\
                                                    ("acc" in i and "con" in i) or\
                                                     ("peri" in i or "pea" in i or "pes" in i))]]

# subset nonsocial valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]

behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]
behavior_df_soc = behavior_df_soc[behavior_df_soc["6_or_more_err_soc"] == 1]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# criteria-based removals
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

# subset only data-related columns
behavior_df_soc[[i for i in behavior_df_soc if ("sub" in i or\
                                                ("acc" in i and "con" in i) or\
                                                ("peri" in i or "pea" in i or "pes" in i))]]

merged_valid_behav_data = behavior_df_soc.merge(behavior_df_nonsoc, on="sub", how="outer")

Full nonsoc-DF length: 239
Removing subjects [3000242, 3000254, 3000255] from nonsocial condition data
New nonsoc-DF length: 238 

[Log] invalid_rt_percent_nonsoc: Replaced 9 outliers (3.78% of data).
[Log] skipped_percent_nonsoc: Replaced 6 outliers (2.52% of data).
Final non-soc-DF length: 223 

Full soc-DF length: 247
Removing subjects [3000047, 3000056, 3000180, 3000182] from social condition data
Removing subjects [3000242, 3000254, 3000255] from social condition data
New soc-DF length: 241 

[Log] invalid_rt_percent_soc: Replaced 6 outliers (2.49% of data).
[Log] skipped_percent_soc: Replaced 9 outliers (3.73% of data).
Final soc-DF length: 227 



In [18]:
merged_valid_behav_data = replace_outliers_with_nan(merged_valid_behav_data, exclude_cols=["sub"])

[Log] invalid_rt_percent_soc: Replaced 7 outliers (2.92% of data).
[Log] skipped_percent_soc: Replaced 9 outliers (3.75% of data).
[Log] acc_soc: Replaced 3 outliers (1.25% of data).
[Log] acc_con_soc: Replaced 3 outliers (1.25% of data).
[Log] rt_con_soc: Replaced 2 outliers (0.83% of data).
[Log] rt_incon_soc: Replaced 1 outliers (0.42% of data).
[Log] rt_corr_soc: Replaced 1 outliers (0.42% of data).
[Log] rt_err_soc: Replaced 6 outliers (2.50% of data).
[Log] rt_incon_log_soc: Replaced 1 outliers (0.42% of data).
[Log] rt_corr_log_soc: Replaced 1 outliers (0.42% of data).
[Log] rt_err_log_soc: Replaced 5 outliers (2.08% of data).
[Log] pes_soc: Replaced 3 outliers (1.25% of data).
[Log] pea_soc: Replaced 2 outliers (0.83% of data).
[Log] peri_rt_soc: Replaced 3 outliers (1.25% of data).
[Log] invalid_rt_percent_nonsoc: Replaced 7 outliers (2.92% of data).
[Log] skipped_percent_nonsoc: Replaced 7 outliers (2.92% of data).
[Log] acc_con_nonsoc: Replaced 1 outliers (0.42% of data).
[L

## REDCap IQS & BBS

In [19]:
id_matrix = pd.Series(list(range(3000000, 3000401)), name = "sub")

iqs_p_data = pd.read_csv(iqs_p_path)
iqs_p_data = iqs_p_data.rename({"record_id" : "sub"}, axis=1)
iqs_p_data["sub"] = iqs_p_data["sub"] - 80000
iqs_p_data = iqs_p_data[
    [i for i in iqs_p_data.columns if (i == "sub" or "scrd" in i)]
]

iqs_ch_data = pd.read_csv(iqs_ch_path)
iqs_ch_data = iqs_ch_data.rename({"record_id" : "sub"}, axis=1)
iqs_ch_data = iqs_ch_data[
    [i for i in iqs_ch_data.columns if (i == "sub" or "scrd" in i)]
]

bbs_p_data = pd.read_csv(bbs_p_path)
bbs_p_data = bbs_p_data.rename({"record_id" : "sub"}, axis=1)
bbs_p_data["sub"] = bbs_p_data["sub"] - 80000
bbs_p_data = bbs_p_data[
    [i for i in bbs_p_data.columns if (i == "sub" or "scrd" in i)]
]

bbs_ch_data = pd.read_csv(bbs_ch_path)
bbs_ch_data = bbs_ch_data.rename({"record_id" : "sub"}, axis=1)
bbs_ch_data = bbs_ch_data[
    [i for i in bbs_ch_data.columns if (i == "sub" or "scrd" in i)]
]

data_frames = [
    id_matrix,
    iqs_p_data,
    iqs_ch_data,
    bbs_p_data,
    bbs_ch_data
]

# the merge is left relative to behav_id
redcap_data = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
# remove IDs that were not present
redcap_data = redcap_data.dropna(how="all", subset = [c for c in redcap_data if c != "sub"]).reset_index(drop=True)
redcap_data.shape[0]

283

In [20]:
# merge spanish columns with corresponding english columns

# 1. Identify Spanish columns
spanish_columns = [col for col in redcap_data.columns if "es_" in col]

for sp_col in spanish_columns:
    # 2. Derive English column name
    # Your logic: replaces "es_" with "_"
    # Example: "demoes_gender" -> "demo_gender"
    eng_col = "_".join(sp_col.split("es_"))
    
    # Safety Check: Ensure the target English column actually exists
    if eng_col not in redcap_data.columns:
        print(f"[Warning] Could not find English match '{eng_col}' for '{sp_col}'. Skipping.")
        continue

    # 3. Vectorized Merge (No loops)
    redcap_data[eng_col] = redcap_data[eng_col].combine_first(redcap_data[sp_col])

# 4. Cleanup
# Only drop the columns we actually processed
redcap_data = redcap_data.drop(columns=spanish_columns, errors='ignore')

In [21]:
# transform state survey data to reflect the order of conditions (alone/observed) within a session
state_surveys = pd.read_csv(bbs_ch_path)
state_surveys = state_surveys.rename({"record_id": "sub"}, axis=1)

state_surveys = state_surveys[
[i for i in state_surveys.columns if (i == "sub" or "selfnowa" in i or "initstatec" in i or "posttaske" in i or "dyada" in i or "initstated" in i or "posttaskf" in i or "dyadb" in i)\
 and ("timestamp" not in i) and ("_complete" not in i)]
]

state_surveys = state_surveys.merge(first_soc, on="sub", how="left")

new_state_survey_df = pd.DataFrame()
# state_surveys[[i for i in state_surveys.columns if ("initstatec" in i or "first_soc" in i)]]
for c, num_items in zip(["initstatec", "posttaske"], [5, 10]):
    for i in range(state_surveys.shape[0]):
        new_state_survey_df.loc[i, "sub"] = state_surveys.loc[i, "sub"]
        for item in range(1, num_items + 1):
            if state_surveys.loc[i, "first_soc"] == 1:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e1"]
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e2"]
            elif state_surveys.loc[i, "first_soc"] == 0:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e2"]
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = state_surveys.loc[i, f"{c}_i{item}_{session}_e1"]
            else:
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_soc"] = np.nan
                new_state_survey_df.loc[i, f"{c}_i{item}_{session}_nonsoc"] = np.nan

new_state_survey_df = new_state_survey_df.dropna(how="all", subset = new_state_survey_df.columns[1:]).reset_index(drop=True)
state_surveys = new_state_survey_df.merge(state_surveys[[i for i in state_surveys.columns if not ("initstatec" in i or "posttaske" in i or "first_soc" in i)]], on="sub", how="left")
redcap_data = redcap_data.merge(state_surveys, on="sub", how="outer")

In [22]:
# --- PART 1: Broad Exclusion (unusable_soc) ---
# Columns to wipe for the "Social" exclusion group
exclude_patterns = ["initstatec", "posttaske", "dyada", "dyadb", "initstated"]
cols_broad = [c for c in redcap_data.columns 
              if np.any([pat in c for pat in exclude_patterns]) 
              and "_nonsoc" not in c]

# Apply NaN for 'unusable_soc' subjects
rows_broad = redcap_data["sub"].isin(unusable_soc)
if cols_broad:
    redcap_data.loc[rows_broad, cols_broad] = np.nan
    print(f"Broad clean: Wiped {len(cols_broad)} columns for subjects {unusable_soc}.")

# --- PART 2: Specific Exclusion (exclude_initstated) ---
# Identify ONLY columns containing 'initstated' (and not nonsocial, if applicable)
cols_init = [c for c in redcap_data.columns 
             if "initstated" in c 
             and "_nonsoc" not in c]

# Apply NaN for 'exclude_initstated' subjects
rows_init = redcap_data["sub"].isin(exclude_initstated)

if cols_init:
    redcap_data.loc[rows_init, cols_init] = np.nan
    print(f"Specific clean: Wiped 'initstated' columns for subjects {exclude_initstated}.")
else:
    print("Warning: No 'initstated' columns found.")


cols_init = [c for c in redcap_data.columns 
             if "initstated" in c 
             and "_nonsoc" not in c]

# Apply NaN for 'exclude_subset_dyadb' subjects
rows_dyadb = redcap_data["sub"].isin(exclude_subset_dyadb)
items_to_exclude = [2, 3, 4, 6, 7, 8]

cols_dyadb = [f"dyadb_i{item}_{session}_e1" for item in items_to_exclude]

if cols_init:
    redcap_data.loc[rows_dyadb, cols_dyadb] = np.nan
    print(f"Specific clean: Wiped 'dyadb' columns for subjects {exclude_subset_dyadb}.")
else:
    print("Warning: No 'dyadb' columns found.")

Broad clean: Wiped 30 columns for subjects [3000047, 3000056, 3000180, 3000182].
Specific clean: Wiped 'initstated' columns for subjects [3000066].
Specific clean: Wiped 'dyadb' columns for subjects [3000361].


In [23]:
redcap_data = replace_outliers_with_nan(redcap_data, exclude_cols=["sub"])

[Log] dersp_scrdCat_s1_r1_e1: Replaced 4 outliers (1.41% of data).
[Log] dersp_scrdNeg_s1_r1_e1: Replaced 2 outliers (0.71% of data).
[Log] edshvsp_scrdEdsEver_s1_r1_e1: Replaced 8 outliers (2.83% of data).
[Log] edshvsp_scrdEdsFreq_s1_r1_e1: Replaced 7 outliers (2.47% of data).
[Log] edshvsp_scrdEdsChron_s1_r1_e1: Replaced 6 outliers (2.12% of data).
[Log] edshvsp_scrdHvsFreq_s1_r1_e1: Replaced 2 outliers (0.71% of data).
[Log] fpep_scrdTotal_s1_r1_e1: Replaced 2 outliers (0.71% of data).
[Log] lsasp_scrdTotal_s1_r1_e1: Replaced 3 outliers (1.06% of data).
[Log] lsasp_scrdTotAnx_s1_r1_e1: Replaced 3 outliers (1.06% of data).
[Log] lsasp_scrdAnxInt_s1_r1_e1: Replaced 3 outliers (1.06% of data).
[Log] lsasp_scrdAnxPer_s1_r1_e1: Replaced 4 outliers (1.41% of data).
[Log] lsasp_scrdTotAvo_s1_r1_e1: Replaced 5 outliers (1.77% of data).
[Log] lsasp_scrdAvoInt_s1_r1_e1: Replaced 3 outliers (1.06% of data).
[Log] lsasp_scrdAvoPer_s1_r1_e1: Replaced 4 outliers (1.41% of data).
[Log] pdsp_scrdF

## DDM

### Subset valid data

In [24]:
# here all valid IDs for DDM are created, which is based on behavioral data
behavior_df = pd.read_csv(behavivor_summary_path)

# now, to perform condition-wise outlier removal, we need to subset nonsocial and social conditions and perform removal separately

# subset non-social valid_data
behavior_df_nonsoc = behavior_df[[col for col in behavior_df.columns if ("_nonsoc" in col or "sub" in col)]]
behavior_df_nonsoc = behavior_df_nonsoc[behavior_df_nonsoc["acc_nonsoc"] >= 0.6]

print(f"Full nonsoc-DF length: {behavior_df_nonsoc.shape[0]}")
print(f"Removing subjects {exclude_id_list} from nonsocial condition data")
behavior_df_nonsoc = behavior_df_nonsoc[~behavior_df_nonsoc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New nonsoc-DF length: {behavior_df_nonsoc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_nonsoc = replace_outliers_with_nan_cols(behavior_df_nonsoc, ["invalid_rt_percent_nonsoc", "skipped_percent_nonsoc"])
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="invalid_rt_percent_nonsoc")
behavior_df_nonsoc = behavior_df_nonsoc.dropna(subset="skipped_percent_nonsoc")
print(f"Final non-soc-DF length: {behavior_df_nonsoc.shape[0]} \n")

valid_behavior_nonsoc = behavior_df_nonsoc["sub"].to_frame()

# subset social valid_data
behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]
behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]

print(f"Full soc-DF length: {behavior_df_soc.shape[0]}")
print(f"Removing subjects {unusable_soc} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(unusable_soc)].reset_index(drop=True)
print(f"Removing subjects {exclude_id_list} from social condition data")
behavior_df_soc = behavior_df_soc[~behavior_df_soc["sub"].isin(exclude_id_list)].reset_index(drop=True)
print(f"New soc-DF length: {behavior_df_soc.shape[0]} \n")

# now we proceed to criteria-based removal
behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
print(f"Final soc-DF length: {behavior_df_soc.shape[0]} \n")

valid_behavior_soc = behavior_df_soc["sub"].to_frame()

Full nonsoc-DF length: 240
Removing subjects [3000242, 3000254, 3000255] from nonsocial condition data
New nonsoc-DF length: 239 

[Log] invalid_rt_percent_nonsoc: Replaced 9 outliers (3.77% of data).
[Log] skipped_percent_nonsoc: Replaced 6 outliers (2.51% of data).
Final non-soc-DF length: 224 

Full soc-DF length: 247
Removing subjects [3000047, 3000056, 3000180, 3000182] from social condition data
Removing subjects [3000242, 3000254, 3000255] from social condition data
New soc-DF length: 241 

[Log] invalid_rt_percent_soc: Replaced 6 outliers (2.49% of data).
[Log] skipped_percent_soc: Replaced 9 outliers (3.73% of data).
Final soc-DF length: 227 



In [25]:
ddm_df = pd.read_csv(ddm_path)
ddm_df = ddm_df.drop("seed", axis=1)

# subset subjects based on valid behavioral flanker data condition-wise
ddm_df = ddm_df[
    (ddm_df['soc'] == 0) | 
    ((ddm_df['soc'] == 1) & (ddm_df['sub'].isin(valid_behavior_soc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

ddm_df = ddm_df[
    (ddm_df['soc'] == 1) | 
    ((ddm_df['soc'] == 0) & (ddm_df['sub'].isin(valid_behavior_nonsoc["sub"])))
].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

# subset subjects who have enough post-error trials
people_passed_cutoff = pd.DataFrame()
cutoff = 16

full_behavior = pd.read_csv(behav_trial_data_path)

counter = 0
for i, sub in enumerate(full_behavior["sub"].unique()):
    # print(sub)
    sub_data = full_behavior[full_behavior["sub"] == sub]
    
    for cond in [0, 1]:
        n_posterr_trials = []
        data = sub_data[sub_data["condition_soc"] == cond]

        n_trials = data[(data["sub"] == sub) & (data["pre_valid_rt"] == 1) & (data["pre_extra_resp"] == 0)\
        & (data["pre_no_resp"] == 0) & (data["pre_congruent"] == 0) & (data["valid_rt"] == 1) & (data["no_resp"] == 0)\
        & (data["pre_accuracy"] == 0)].shape[0]

        if n_trials >= cutoff:
            people_passed_cutoff.loc[counter, "sub"] = sub
            people_passed_cutoff.loc[counter, "soc"] = cond
            people_passed_cutoff.loc[counter, "n_posterr_trial"] = n_trials
            counter += 1

ddm_df = ddm_df.merge(people_passed_cutoff[['sub', 'soc']], on=['sub', 'soc'], how='inner')
# remove subjects with bad fit
ddm_df = ddm_df[ddm_df["fitStat"] <= 200].reset_index(drop=True)
print(f"Full DF length: {ddm_df.shape[0]}")

Full DF length: 946
Full DF length: 902
Full DF length: 792


### Difference scores (Posterror - Postcorrect)

In [26]:
# convert data to wide format

# ddm_df_soc = ddm_df[ddm_df["soc"] == 1]
# ddm_df_nonsoc = ddm_df[ddm_df["soc"] == 0]

def get_condition_suffix(row):
    acc_str = 'postcorr' if row['pre_accuracy'] == 1 else 'posterr'
    soc_str = 'soc' if row['soc'] == 1 else 'nonsoc'
    return f"{acc_str}_{soc_str}"

ddm_df['condition'] = ddm_df.apply(get_condition_suffix, axis=1)

df_pivoted = ddm_df.pivot(index='sub', columns='condition')

df_pivoted.columns = [f"{col[0]}_{col[1]}" for col in df_pivoted.columns]

df_pivoted.reset_index(inplace=True)

cols_to_drop = [c for c in df_pivoted.columns if c.startswith('pre_accuracy') or c.startswith('soc_')]
ddm_df = df_pivoted.drop(columns=cols_to_drop, errors='ignore')

ddm_df = replace_outliers_with_nan(ddm_df, exclude_cols=["sub"])

[Log] a_postcorr_soc: Replaced 1 outliers (0.45% of data).
[Log] a_posterr_nonsoc: Replaced 2 outliers (0.89% of data).
[Log] a_posterr_soc: Replaced 2 outliers (0.89% of data).
[Log] ter_postcorr_soc: Replaced 1 outliers (0.45% of data).
[Log] ter_posterr_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] ter_posterr_soc: Replaced 3 outliers (1.34% of data).
[Log] p_postcorr_soc: Replaced 3 outliers (1.34% of data).
[Log] rd_postcorr_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] fitStat_postcorr_nonsoc: Replaced 4 outliers (1.79% of data).
[Log] fitStat_postcorr_soc: Replaced 1 outliers (0.45% of data).
[Log] fitStat_posterr_nonsoc: Replaced 3 outliers (1.34% of data).
[Log] fitStat_posterr_soc: Replaced 5 outliers (2.23% of data).
[Log] iterNum_postcorr_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] iterNum_postcorr_soc: Replaced 2 outliers (0.89% of data).
[Log] iterNum_posterr_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] iterNum_posterr_soc: Replaced 1 outliers (0.45

In [27]:
# create attentional ratio columns
# Identify all 'Left' columns to drive the loop
rd_cols = [c for c in ddm_df.columns if 'rd_' in c]

for rd_col in rd_cols:
    # Construct the corresponding 'Right' column name
    sda_col = rd_col.replace('rd_', 'sda_')
    
    # Check if the Right pair exists
    if sda_col in ddm_df.columns:
        # Create new name: 
        # 1. Replace '_L_' with '_' (ICPS_soc_early_DLPFC_L_err -> ICPS_soc_early_DLPFC_err)
        # 2. Append '_collapsed'
        new_col = rd_col.replace('rd_', 'reversed_ratio_')
        
        # Calculate mean (NaN + Value = NaN)
        ddm_df[new_col] = (ddm_df[sda_col] / ddm_df[rd_col]) * (-1)

In [28]:
# compute difference scores
# Parameters and conditions to iterate over
params = ['a', 'ter', 'p', 'sda', 'rd', 'reversed_ratio']
conditions = ['soc', 'nonsoc']

for param in params:
    for cond in conditions:
        # Construct existing column names
        posterr_col = f'{param}_posterr_{cond}'
        postcorr_col = f'{param}_postcorr_{cond}'
        
        # Define new difference column name
        # e.g., a_posterr_min_postcorr_soc
        diff_col = f'{param}_diff_{cond}'
        
        # Vectorized subtraction
        # (posterr - postcorr)
        ddm_df[diff_col] = ddm_df[posterr_col] - ddm_df[postcorr_col]

In [29]:
ddm_df = replace_outliers_with_nan_cols(ddm_df, columns_to_check = [c for c in ddm_df if "_diff_" in c])

[Log] a_diff_soc: No outliers found.
[Log] a_diff_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] ter_diff_soc: Replaced 1 outliers (0.45% of data).
[Log] ter_diff_nonsoc: Replaced 1 outliers (0.45% of data).
[Log] p_diff_soc: Replaced 2 outliers (0.89% of data).
[Log] p_diff_nonsoc: Replaced 2 outliers (0.89% of data).
[Log] sda_diff_soc: No outliers found.
[Log] sda_diff_nonsoc: No outliers found.
[Log] rd_diff_soc: No outliers found.
[Log] rd_diff_nonsoc: No outliers found.
[Log] reversed_ratio_diff_soc: No outliers found.
[Log] reversed_ratio_diff_nonsoc: No outliers found.


## Final merge

In [30]:
id_matrix = pd.Series(list(range(3000000, 3000401)), name = "sub")

data_frames = [
    id_matrix,
    age_data,
    sex_data,
    first_soc,
    dp_mode,
    merged_valid_eeg_data,
    merged_valid_behav_data,
    ddm_df,
    redcap_data
]

merged_df = reduce(lambda left, right: pd.merge(left, right, on="sub", how='left'), data_frames)
print(merged_df.shape)

# remove IDs that were not present
merged_df = merged_df.dropna(how="all", subset = [c for c in merged_df if c != "sub"]).reset_index(drop=True)
print(merged_df.shape)

date_time = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
merged_df.to_csv(f"thrive_wide_{session}_{date_time}.csv", index=False)

(401, 448)
(283, 448)


In [37]:
for df in data_frames[1:]:
    # print(list(df.columns[0]))
    display(df[df["sub"] == 3000196])

,sub,age_m
158,3000196,165.0


,sub,sex
158,3000196,2


,sub,first_soc
146,3000196,0


,sub,dp_inperson
150,3000196.0,1.0


,sub,ERN_soc,CRN_soc,ERN_soc_laplacian,CRN_soc_laplacian,power_soc_early_err,power_soc_late_err,power_soc_early_corr,power_soc_late_corr,ITPS_soc_early_err,ITPS_soc_late_err,ITPS_soc_early_corr,ITPS_soc_late_corr,ICPS_soc_early_DLPFC_L_err,ICPS_soc_early_DLPFC_R_err,ICPS_soc_early_OCC_L_err,ICPS_soc_early_OCC_R_err,ICPS_soc_early_MOTOR_L_err,ICPS_soc_early_MOTOR_R_err,ICPS_soc_late_DLPFC_L_err,ICPS_soc_late_DLPFC_R_err,ICPS_soc_late_OCC_L_err,ICPS_soc_late_OCC_R_err,ICPS_soc_late_MOTOR_L_err,ICPS_soc_late_MOTOR_R_err,ICPS_soc_early_DLPFC_L_corr,ICPS_soc_early_DLPFC_R_corr,ICPS_soc_early_OCC_L_corr,ICPS_soc_early_OCC_R_corr,ICPS_soc_early_MOTOR_L_corr,ICPS_soc_early_MOTOR_R_corr,ICPS_soc_late_DLPFC_L_corr,ICPS_soc_late_DLPFC_R_corr,ICPS_soc_late_OCC_L_corr,ICPS_soc_late_OCC_R_corr,ICPS_soc_late_MOTOR_L_corr,ICPS_soc_late_MOTOR_R_corr,ERN_nonsoc,CRN_nonsoc,ERN_nonsoc_laplacian,CRN_nonsoc_laplacian,power_nonsoc_early_err,power_nonsoc_late_err,power_nonsoc_early_corr,power_nonsoc_late_corr,ITPS_nonsoc_early_err,ITPS_nonsoc_late_err,ITPS_nonsoc_early_corr,ITPS_nonsoc_late_corr,ICPS_nonsoc_early_DLPFC_L_err,ICPS_nonsoc_early_DLPFC_R_err,ICPS_nonsoc_early_OCC_L_err,ICPS_nonsoc_early_OCC_R_err,ICPS_nonsoc_early_MOTOR_L_err,ICPS_nonsoc_early_MOTOR_R_err,ICPS_nonsoc_late_DLPFC_L_err,ICPS_nonsoc_late_DLPFC_R_err,ICPS_nonsoc_late_OCC_L_err,ICPS_nonsoc_late_OCC_R_err,ICPS_nonsoc_late_MOTOR_L_err,ICPS_nonsoc_late_MOTOR_R_err,ICPS_nonsoc_early_DLPFC_L_corr,ICPS_nonsoc_early_DLPFC_R_corr,ICPS_nonsoc_early_OCC_L_corr,ICPS_nonsoc_early_OCC_R_corr,ICPS_nonsoc_early_MOTOR_L_corr,ICPS_nonsoc_early_MOTOR_R_corr,ICPS_nonsoc_late_DLPFC_L_corr,ICPS_nonsoc_late_DLPFC_R_corr,ICPS_nonsoc_late_OCC_L_corr,ICPS_nonsoc_late_OCC_R_corr,ICPS_nonsoc_late_MOTOR_L_corr,ICPS_nonsoc_late_MOTOR_R_corr,power_soc_early_diff,power_soc_late_diff,ITPS_soc_early_diff,ITPS_soc_late_diff,ICPS_soc_early_DLPFC_L_diff,ICPS_soc_early_DLPFC_R_diff,ICPS_soc_early_OCC_L_diff,ICPS_soc_early_OCC_R_diff,ICPS_soc_early_MOTOR_L_diff,ICPS_soc_early_MOTOR_R_diff,ICPS_soc_late_DLPFC_L_diff,ICPS_soc_late_DLPFC_R_diff,ICPS_soc_late_OCC_L_diff,ICPS_soc_late_OCC_R_diff,ICPS_soc_late_MOTOR_L_diff,ICPS_soc_late_MOTOR_R_diff,power_nonsoc_early_diff,power_nonsoc_late_diff,ITPS_nonsoc_early_diff,ITPS_nonsoc_late_diff,ICPS_nonsoc_early_DLPFC_L_diff,ICPS_nonsoc_early_DLPFC_R_diff,ICPS_nonsoc_early_OCC_L_diff,ICPS_nonsoc_early_OCC_R_diff,ICPS_nonsoc_early_MOTOR_L_diff,ICPS_nonsoc_early_MOTOR_R_diff,ICPS_nonsoc_late_DLPFC_L_diff,ICPS_nonsoc_late_DLPFC_R_diff,ICPS_nonsoc_late_OCC_L_diff,ICPS_nonsoc_late_OCC_R_diff,ICPS_nonsoc_late_MOTOR_L_diff,ICPS_nonsoc_late_MOTOR_R_diff,ERN_min_CRN_soc,ERN_min_CRN_nonsoc,ERN_min_CRN_soc_laplacian,ERN_min_CRN_nonsoc_laplacian,ICPS_soc_early_DLPFC_err_collapsed,ICPS_soc_early_OCC_err_collapsed,ICPS_soc_early_MOTOR_err_collapsed,ICPS_soc_late_DLPFC_err_collapsed,ICPS_soc_late_OCC_err_collapsed,ICPS_soc_late_MOTOR_err_collapsed,ICPS_soc_early_DLPFC_corr_collapsed,ICPS_soc_early_OCC_corr_collapsed,ICPS_soc_early_MOTOR_corr_collapsed,ICPS_soc_late_DLPFC_corr_collapsed,ICPS_soc_late_OCC_corr_collapsed,ICPS_soc_late_MOTOR_corr_collapsed,ICPS_nonsoc_early_DLPFC_err_collapsed,ICPS_nonsoc_early_OCC_err_collapsed,ICPS_nonsoc_early_MOTOR_err_collapsed,ICPS_nonsoc_late_DLPFC_err_collapsed,ICPS_nonsoc_late_OCC_err_collapsed,ICPS_nonsoc_late_MOTOR_err_collapsed,ICPS_nonsoc_early_DLPFC_corr_collapsed,ICPS_nonsoc_early_OCC_corr_collapsed,ICPS_nonsoc_early_MOTOR_corr_collapsed,ICPS_nonsoc_late_DLPFC_corr_collapsed,ICPS_nonsoc_late_OCC_corr_collapsed,ICPS_nonsoc_late_MOTOR_corr_collapsed,ICPS_soc_early_DLPFC_diff_collapsed,ICPS_soc_early_OCC_diff_collapsed,ICPS_soc_early_MOTOR_diff_collapsed,ICPS_soc_late_DLPFC_diff_collapsed,ICPS_soc_late_OCC_diff_collapsed,ICPS_soc_late_MOTOR_diff_collapsed,ICPS_nonsoc_early_DLPFC_diff_collapsed,ICPS_nonsoc_early_OCC_diff_collapsed,ICPS_nonsoc_early_MOTOR_diff_collapsed,ICPS_nonsoc_late_DLPFC_diff_collapsed,ICPS_nonsoc_late_OCC_diff_collapsed,ICPS_no

,sub,n_trials_soc,invalid_rt_percent_soc,skipped_percent_soc,acc_soc,acc_con_soc,acc_incon_soc,rt_con_soc,rt_incon_soc,rt_corr_soc,rt_err_soc,rt_con_log_soc,rt_incon_log_soc,rt_corr_log_soc,rt_err_log_soc,pes_soc,pea_soc,peri_acc_soc,peri_rt_soc,6_or_more_err_soc,n_trials_nonsoc,invalid_rt_percent_nonsoc,skipped_percent_nonsoc,acc_nonsoc,acc_con_nonsoc,acc_incon_nonsoc,rt_con_nonsoc,rt_incon_nonsoc,rt_corr_nonsoc,rt_err_nonsoc,rt_con_log_nonsoc,rt_incon_log_nonsoc,rt_corr_log_nonsoc,rt_err_log_nonsoc,pes_nonsoc,pea_nonsoc,peri_acc_nonsoc,peri_rt_nonsoc,6_or_more_err_nonsoc
132,3000196,400.0,0.5,0.5,0.907,0.985,0.829,399.301,469.754,469.754,396.307,-927.574,-767.773,-767.773,-973.776,-0.01626,-0.0739,0.10599,0.20835,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,sub,a_postcorr_nonsoc,a_postcorr_soc,a_posterr_nonsoc,a_posterr_soc,ter_postcorr_nonsoc,ter_postcorr_soc,ter_posterr_nonsoc,ter_posterr_soc,p_postcorr_nonsoc,p_postcorr_soc,p_posterr_nonsoc,p_posterr_soc,rd_postcorr_nonsoc,rd_postcorr_soc,rd_posterr_nonsoc,rd_posterr_soc,sda_postcorr_nonsoc,sda_postcorr_soc,sda_posterr_nonsoc,sda_posterr_soc,fitStat_postcorr_nonsoc,fitStat_postcorr_soc,fitStat_posterr_nonsoc,fitStat_posterr_soc,iterNum_postcorr_nonsoc,iterNum_postcorr_soc,iterNum_posterr_nonsoc,iterNum_posterr_soc,reversed_ratio_postcorr_nonsoc,reversed_ratio_postcorr_soc,reversed_ratio_posterr_nonsoc,reversed_ratio_posterr_soc,a_diff_soc,a_diff_nonsoc,ter_diff_soc,ter_diff_nonsoc,p_diff_soc,p_diff_nonsoc,sda_diff_soc,sda_diff_nonsoc,rd_diff_soc,rd_diff_nonsoc,reversed_ratio_diff_soc,reversed_ratio_diff_nonsoc
120,3000196.0,NaN,0.121473,NaN,0.086342,NaN,0.275548,NaN,0.275076,NaN,0.484104,NaN,0.326362,NaN,0.020646,NaN,0.012058,NaN,1.840627,NaN,1.657858,NaN,46.128023,NaN,14.44689,NaN,99.0,NaN,83.0,NaN,-89.150123,NaN,-137.488472,-0.035131,NaN,-0.000472,NaN,-0.157742,NaN,-0.182768,NaN,-0.008588,NaN,-48.338348,NaN


,sub,bfnep_b_scrdTotal_s1_r1_e1,dersp_scrdCat_s1_r1_e1,dersp_scrdNeg_s1_r1_e1,dersp_scrdAtt_s1_r1_e1,dersp_scrdDis_s1_r1_e1,edshvsp_scrdEdsEver_s1_r1_e1,edshvsp_scrdEdsFreq_s1_r1_e1,edshvsp_scrdEdsChron_s1_r1_e1,edshvsp_scrdHvsEver_s1_r1_e1,edshvsp_scrdHvsFreq_s1_r1_e1,esip_scrdTotal_s1_r1_e1,fpep_scrdTotal_s1_r1_e1,lsasp_scrdTotal_s1_r1_e1,lsasp_scrdTotAnx_s1_r1_e1,lsasp_scrdAnxInt_s1_r1_e1,lsasp_scrdAnxPer_s1_r1_e1,lsasp_scrdTotAvo_s1_r1_e1,lsasp_scrdAvoInt_s1_r1_e1,lsasp_scrdAvoPer_s1_r1_e1,pdsp_scrdFem_s1_r1_e1,pdsp_scrdMal_s1_r1_e1,psbp_scrdTotal_s1_r1_e1,pscei_scrdTotal_s1_r1_e1,rcadsp_scrdTotal_s1_r1_e1,rcadsp_scrdAnx_s1_r1_e1,rcadsp_scrdDep_s1_r1_e1,rpeqp_scrdOv_s1_r1_e1,rpeqp_scrdRel_s1_r1_e1,rpeqp_scrdRep_s1_r1_e1,rpeqp_scrdPro_s1_r1_e1,spaip_scrdTotal_s1_r1_e1,ats_scrdTotal_s1_r1_e1,ats_scrdStd_s1_r1_e1,ats_scrdGen_s1_r1_e1,ats_scrdCrit_s1_r1_e1,bfne_b_scrdTotal_s1_r1_e1,casi_scrdTotal_s1_r1_e1,edshvsc_scrdEdsEver_s1_r1_e1,edshvsc_scrdEdsFreq_s1_r1_e1,edshvsc_scrdEdsChron_s1_r1_e1,edshvsc_scrdHvsEver_s1_r1_e1,edshvsc_scrdHvsFreq_s1_r1_e1,erqca_scrdCogRea_s1_r1_e1,erqca_scrdExpSup_s1_r1_e1,esi_scrdTotal_s1_r1_e1,fpe_scrdTotal_s1_r1_e1,lsasca_scrdTotal_s1_r1_e1,lsasca_scrdTotAnx_s1_r1_e1,lsasca_scrdAnxInt_s1_r1_e1,lsasca_scrdAnxPer_s1_r1_e1,lsasca_scrdTotAvo_s1_r1_e1,lsasca_scrdAvoInt_s1_r1_e1,lsasca_scrdAvoPer_s1_r1_e1,mss_scrdSDS_s1_r1_e1,ocic_scrdTotal_s1_r1_e1,ocic_scrdDoubt_s1_r1_e1,ocic_scrdObs_s1_r1_e1,ocic_scrdHoard_s1_r1_e1,ocic_scrdWash_s1_r1_e1,ocic_scrdOrder_s1_r1_e1,ocic_scrdNeu_s1_r1_e1,oopjr_scrdTotal_s1_r1_e1,pds_scrdFem_s1_r1_e1,pds_scrdMal_s1_r1_e1,pints_scrdTotal_s1_r1_e1,psb_scrdTotal_s1_r1_e1,rcads_scrdTotal_s1_r1_e1,rcads_scrdAnx_s1_r1_e1,rcads_scrdDep_s1_r1_e1,rpeq_scrdOv_s1_r1_e1,rpeq_scrdRel_s1_r1_e1,rpeq_scrdRep_s1_r1_e1,rpeq_scrdPro_s1_r1_e1,sassy_scrdTotal_s1_r1_e1,sassy_scrdSocRej_s1_r1_e1,sassy_scrdBlame_s1_r1_e1,socrew_scrdTotal_s1_r1_e1,socrew_scrdLike_s1_r1_e1,socrew_scrdWant_s1_r1_e1,socrew_scrdEff_s1_r1_e1,spaic_scrdTotal_s1_r1_e1,swcq_scrdAca_s1_r1_e1,swcq_scrdSoc_s1_r1_e1,swcq_scrdTotal_s1_r1_e1,tsis_scrdSIP_s1_r1_e1,adexi_b_scrdWm_s1_r1_e1,adexi_b_scrdInh_s1_r1_e1,adexi_b_scrdTotal_s1_r1_e1,bfne_b_parent_scrdTotal_s1_r1_e1,bfne_b_parent_scrdTotal_s1_r1_e1.1,eatq_scrdAC_s1_r1_e1,eatq_scrdAff_s1_r1_e1,eatq_scrdAgg_s1_r1_e1,eatq_scrdAtt_s1_r1_e1,eatq_scrdDM_s1_r1_e1,eatq_scrdFear_s1_r1_e1,eatq_scrdFrus_s1_r1_e1,erq_scrdCogRea_s1_r1_e1,erq_scrdExpSup_s1_r1_e1,fasap_scrdAcc_s1_r1_e1,fasap_scrdPart_s1_r1_e1,fasap_scrdMod_s1_r1_e1,fasap_scrdDist_s1_r1_e1,fasap_scrdCons_s1_r1_e1,fpe_parent_scrdTotal_s1_r1_e1,fpe_parent_scrdTotal_s1_r1_e1.1,masi_b_parent_scrdTotal_s1_r1_e1,masi_b_parent_scrdAccPro_s1_r1_e1,masi_b_parent_scrdAccCon_s1_r1_e1,masi_b_parent_scrdTotal_s1_r1_e1.1,masi_b_parent_scrdAccPro_s1_r1_e1.1,masi_b_parent_scrdAccCon_s1_r1_e1.1,phq8_scrdTotal_s1_r1_e1,pints_parent_scrdTotal_s1_r1_e1,pints_parent_scrdTotal_s1_r1_e1.1,pss_scrdTotal_s1_r1_e1,rpbip_scrdAcc_s1_r1_e1,rpbip_scrdPsyC_s1_r1_e1,rsrip_scrdTotal_s1_r1_e1,rsrip_scrdSocSch_s1_r1_e1,rsrip_scrdFear_s1_r1_e1,scaared_b_scrdTotal_s1_r1_e1,scaared_b_scrdPaSo_s1_r1_e1,scaared_b_scrdGA_s1_r1_e1,scaared_b_scrdSep_s1_r1_e1,scaared_b_scrdSoc_s1_r1_e1,scaredp_scrdTotal_s1_r1_e1,scaredp_scrdPan_s1_r1_e1,scaredp_scrdGA_s1_r1_e1,scaredp_scrdSep_s1_r1_e1,scaredp_scrdSA_s1_r1_e1,scaredp_scrdAvo_s1_r1_e1,texip_b_scrdWm_s1_r1_e1,texip_b_scrdInh_s1_r1_e1,via_parent_scrdHer_s1_r1_e1,via_parent_scrdMain_s1_r1_e1,via_parent_scrdHer_s1_r1_e1.1,via_parent_scrdMain_s1_r1_e1.1,abq_scrdTotal_s1_r1_e1,abq_scrdEng_s1_r1_e1,abq_scrdDis_s1_r1_e1,epepq15_scrdTotal_s1_r1_e1,epepq15_scrdCI_s1_r1_e1,epepq15_scrdNS_s1_r1_e1,epepq15_scrdTP_s1_r1_e1,masi_b_scrdTotal_s1_r1_e1,masi_b_scrdAccPro_s1_r1_e1,masi_b_scrdAccCon_s1_r1_e1,rpbic_scrdAcc_s1_r1_e1,rpbic_scrdPsyC_s1_r1_e1,texi_b_scrdWm_s1_r1_e1,texi_b_scrdInh_s1_r1_e1,via_scrdHer_s1_r1_e1,via_scrdMain_s1_r1_e1,initstatec_i1_s1_r1_soc,initstatec_i1_s1_r1_nonsoc,initstatec_i2_s1_r1_soc,initstatec_i2_s1_r1_no

In [31]:
import pandas as pd

# Load data
df = pd.read_csv(find_newest_file(f"thrive_wide_{session}_*.csv"))

# 1. Define Column Categories
id_cols = ['sub', 'age_m', 'sex', 'first_soc', 'dp_inperson']
redcap_cols = [c for c in df if session in c]

# Social-Only Variables (Repeated for acc=0 and acc=1)
# These vary by social condition but are NOT split by accuracy in the wide format
social_only_bases_input = [
    'n_trials', 'invalid_rt_percent', 'skipped_percent', 'acc', 'acc_con', 
    'acc_incon', 'rt_con', 'rt_incon', 'rt_con_log', 'rt_incon_log', 
    'rt_corr_log', 'rt_err_log', 'pes', 'pea', 'peri_acc', 'peri_rt', 
    '6_or_more_err'
]

col_map = {}

for c in df.columns:
    # ID Columns
    if c in id_cols:
        col_map[c] = {'type': 'id'}
        continue
        
    mapped = False
    
    # --- 1. Check Explicit Social-Only List ---
    for base in social_only_bases_input:
        if c == f"{base}_soc":
            # Rename 'acc' to 'accuracy_score' to avoid conflict with index 'acc'
            b = 'accuracy_score' if base == 'acc' else base
            col_map[c] = {'type': 'social', 'base': b, 'soc': 1}
            mapped = True
            break
        elif c == f"{base}_nonsoc":
            b = 'accuracy_score' if base == 'acc' else base
            col_map[c] = {'type': 'social', 'base': b, 'soc': 0}
            mapped = True
            break
    if mapped: continue
    
    # --- 2. Check Social-Only Patterns (diff, ERN_min_CRN) ---
    # Matches 'a_diff_soc', 'power_soc_early_diff', etc.
    if 'diff' in c or 'ERN_min_CRN' in c:
        if '_soc' in c:
            base = c.replace('_soc', '') 
            col_map[c] = {'type': 'social', 'base': base, 'soc': 1}
        elif '_nonsoc' in c:
            base = c.replace('_nonsoc', '')
            col_map[c] = {'type': 'social', 'base': base, 'soc': 0}
        mapped = True
        continue
    
    # --- 3. Check Post-Error / Post-Correct (NEW) ---
    # Handled before generic _err/_corr to ensure correct base extraction
    if '_posterr' in c or '_postcorr' in c:
        is_err = '_posterr' in c
        
        if '_soc' in c:
            soc = 1
            # Remove suffixes to get base (e.g., 'a_posterr_soc' -> 'a')
            base = c.replace('_soc', '').replace('_posterr', '').replace('_postcorr', '')
        elif '_nonsoc' in c:
            soc = 0
            base = c.replace('_nonsoc', '').replace('_posterr', '').replace('_postcorr', '')
        else:
            continue
            
        col_map[c] = {'type': 'crossed', 'base': base, 'soc': soc, 'acc': 0 if is_err else 1}
        continue

    # --- 4. Check ERN / CRN (Crossed) ---
    if c.startswith('ERN') or c.startswith('CRN'):
        is_ern = c.startswith('ERN')
        
        # Determine social condition
        if '_soc' in c:
            soc = 1
            suffix = c.replace('ERN', '').replace('CRN', '').replace('_soc', '')
        elif '_nonsoc' in c:
            soc = 0
            suffix = c.replace('ERN', '').replace('CRN', '').replace('_nonsoc', '')
        else:
            continue

        # Clean suffix to get base (empty suffix -> 'amplitude')
        base = suffix.strip('_')
        if base == '': base = 'amplitude'
        if base == 'laplacian': base = 'laplacian'
        
        col_map[c] = {'type': 'crossed', 'base': base, 'soc': soc, 'acc': 0 if is_ern else 1}
        continue

    # --- 5. Check Generic _err / _corr Suffix (Crossed) ---
    if '_err' in c or '_corr' in c:
        is_err = '_err' in c
        if '_soc' in c:
            soc = 1
            base = c.replace('_soc', '').replace('_err', '').replace('_corr', '')
        elif '_nonsoc' in c:
            soc = 0
            base = c.replace('_nonsoc', '').replace('_err', '').replace('_corr', '')
        else:
            continue
        
        col_map[c] = {'type': 'crossed', 'base': base, 'soc': soc, 'acc': 0 if is_err else 1}
        continue

# --- Transformation ---

melted = df.melt(id_vars=id_cols, var_name='original_col', value_name='value')
meta = melted['original_col'].map(col_map)
melted = melted[meta.notna()]
meta = meta[meta.notna()]

melted['base'] = meta.apply(lambda x: x.get('base'))
melted['soc'] = meta.apply(lambda x: x.get('soc'))
melted['acc_spec'] = meta.apply(lambda x: x.get('acc'))
melted['type'] = meta.apply(lambda x: x.get('type'))

# Split rows: Duplicate Social-only rows
social_rows = melted[melted['type'] == 'social'].copy()
crossed_rows = melted[melted['type'] == 'crossed'].copy()

social_0 = social_rows.copy()
social_0['acc'] = 0
social_1 = social_rows.copy()
social_1['acc'] = 1

# Crossed rows use their specific accuracy
crossed_rows['acc'] = crossed_rows['acc_spec']

final_long = pd.concat([social_0, social_1, crossed_rows], ignore_index=True)

# Pivot
pivot_index = id_cols + ['soc', 'acc']
pivot_cols = 'base'
pivot_values = 'value'

df_long = final_long.pivot_table(index=pivot_index, columns=pivot_cols, values=pivot_values, aggfunc='first')

# Cleanup Index
df_long.index.names = [n + '_idx' for n in df_long.index.names] # Rename to avoid collision
df_long = df_long.reset_index()
df_long.rename(columns={'soc_idx': 'soc', 'acc_idx': 'acc'}, inplace=True)
df_long.rename(columns={c: c.replace('_idx', '') for c in df_long.columns if c.endswith('_idx')}, inplace=True)
df_long = df_long.merge(df[['sub'] + redcap_cols].drop_duplicates('sub'), on='sub', how='left')
# Ensure Integer Types
df_long['soc'] = df_long['soc'].astype(int)
df_long['acc'] = df_long['acc'].astype(int)

# Save
df_long.to_csv(f"thrive_long_{session}_{date_time}.csv", index=False)
# print("Transformation complete. Columns:", df_long.columns.tolist())

The newest file is: thrive_wide_s1_r1_17_01_2026_19_46_00.csv


## Checks

In [36]:
# Display all columns
pd.set_option('display.max_columns', None)

# Display all rows
pd.set_option('display.max_rows', None)

merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]

,sub,ERN_soc,CRN_soc,ERN_soc_laplacian,CRN_soc_laplacian,power_soc_early_err,power_soc_late_err,power_soc_early_corr,power_soc_late_corr,ITPS_soc_early_err,ITPS_soc_late_err,ITPS_soc_early_corr,ITPS_soc_late_corr,ICPS_soc_early_DLPFC_L_err,ICPS_soc_early_DLPFC_R_err,ICPS_soc_early_OCC_L_err,ICPS_soc_early_OCC_R_err,ICPS_soc_early_MOTOR_L_err,ICPS_soc_early_MOTOR_R_err,ICPS_soc_late_DLPFC_L_err,ICPS_soc_late_DLPFC_R_err,ICPS_soc_late_OCC_L_err,ICPS_soc_late_OCC_R_err,ICPS_soc_late_MOTOR_L_err,ICPS_soc_late_MOTOR_R_err,ICPS_soc_early_DLPFC_L_corr,ICPS_soc_early_DLPFC_R_corr,ICPS_soc_early_OCC_L_corr,ICPS_soc_early_OCC_R_corr,ICPS_soc_early_MOTOR_L_corr,ICPS_soc_early_MOTOR_R_corr,ICPS_soc_late_DLPFC_L_corr,ICPS_soc_late_DLPFC_R_corr,ICPS_soc_late_OCC_L_corr,ICPS_soc_late_OCC_R_corr,ICPS_soc_late_MOTOR_L_corr,ICPS_soc_late_MOTOR_R_corr,ERN_nonsoc,CRN_nonsoc,ERN_nonsoc_laplacian,CRN_nonsoc_laplacian,power_nonsoc_early_err,power_nonsoc_late_err,power_nonsoc_early_corr,power_nonsoc_late_corr,ITPS_nonsoc_early_err,ITPS_nonsoc_late_err,ITPS_nonsoc_early_corr,ITPS_nonsoc_late_corr,ICPS_nonsoc_early_DLPFC_L_err,ICPS_nonsoc_early_DLPFC_R_err,ICPS_nonsoc_early_OCC_L_err,ICPS_nonsoc_early_OCC_R_err,ICPS_nonsoc_early_MOTOR_L_err,ICPS_nonsoc_early_MOTOR_R_err,ICPS_nonsoc_late_DLPFC_L_err,ICPS_nonsoc_late_DLPFC_R_err,ICPS_nonsoc_late_OCC_L_err,ICPS_nonsoc_late_OCC_R_err,ICPS_nonsoc_late_MOTOR_L_err,ICPS_nonsoc_late_MOTOR_R_err,ICPS_nonsoc_early_DLPFC_L_corr,ICPS_nonsoc_early_DLPFC_R_corr,ICPS_nonsoc_early_OCC_L_corr,ICPS_nonsoc_early_OCC_R_corr,ICPS_nonsoc_early_MOTOR_L_corr,ICPS_nonsoc_early_MOTOR_R_corr,ICPS_nonsoc_late_DLPFC_L_corr,ICPS_nonsoc_late_DLPFC_R_corr,ICPS_nonsoc_late_OCC_L_corr,ICPS_nonsoc_late_OCC_R_corr,ICPS_nonsoc_late_MOTOR_L_corr,ICPS_nonsoc_late_MOTOR_R_corr,power_soc_early_diff,power_soc_late_diff,ITPS_soc_early_diff,ITPS_soc_late_diff,ICPS_soc_early_DLPFC_L_diff,ICPS_soc_early_DLPFC_R_diff,ICPS_soc_early_OCC_L_diff,ICPS_soc_early_OCC_R_diff,ICPS_soc_early_MOTOR_L_diff,ICPS_soc_early_MOTOR_R_diff,ICPS_soc_late_DLPFC_L_diff,ICPS_soc_late_DLPFC_R_diff,ICPS_soc_late_OCC_L_diff,ICPS_soc_late_OCC_R_diff,ICPS_soc_late_MOTOR_L_diff,ICPS_soc_late_MOTOR_R_diff,power_nonsoc_early_diff,power_nonsoc_late_diff,ITPS_nonsoc_early_diff,ITPS_nonsoc_late_diff,ICPS_nonsoc_early_DLPFC_L_diff,ICPS_nonsoc_early_DLPFC_R_diff,ICPS_nonsoc_early_OCC_L_diff,ICPS_nonsoc_early_OCC_R_diff,ICPS_nonsoc_early_MOTOR_L_diff,ICPS_nonsoc_early_MOTOR_R_diff,ICPS_nonsoc_late_DLPFC_L_diff,ICPS_nonsoc_late_DLPFC_R_diff,ICPS_nonsoc_late_OCC_L_diff,ICPS_nonsoc_late_OCC_R_diff,ICPS_nonsoc_late_MOTOR_L_diff,ICPS_nonsoc_late_MOTOR_R_diff,ERN_min_CRN_soc,ERN_min_CRN_nonsoc,ERN_min_CRN_soc_laplacian,ERN_min_CRN_nonsoc_laplacian,ICPS_soc_early_DLPFC_err_collapsed,ICPS_soc_early_OCC_err_collapsed,ICPS_soc_early_MOTOR_err_collapsed,ICPS_soc_late_DLPFC_err_collapsed,ICPS_soc_late_OCC_err_collapsed,ICPS_soc_late_MOTOR_err_collapsed,ICPS_soc_early_DLPFC_corr_collapsed,ICPS_soc_early_OCC_corr_collapsed,ICPS_soc_early_MOTOR_corr_collapsed,ICPS_soc_late_DLPFC_corr_collapsed,ICPS_soc_late_OCC_corr_collapsed,ICPS_soc_late_MOTOR_corr_collapsed,ICPS_nonsoc_early_DLPFC_err_collapsed,ICPS_nonsoc_early_OCC_err_collapsed,ICPS_nonsoc_early_MOTOR_err_collapsed,ICPS_nonsoc_late_DLPFC_err_collapsed,ICPS_nonsoc_late_OCC_err_collapsed,ICPS_nonsoc_late_MOTOR_err_collapsed,ICPS_nonsoc_early_DLPFC_corr_collapsed,ICPS_nonsoc_early_OCC_corr_collapsed,ICPS_nonsoc_early_MOTOR_corr_collapsed,ICPS_nonsoc_late_DLPFC_corr_collapsed,ICPS_nonsoc_late_OCC_corr_collapsed,ICPS_nonsoc_late_MOTOR_corr_collapsed,ICPS_soc_early_DLPFC_diff_collapsed,ICPS_soc_early_OCC_diff_collapsed,ICPS_soc_early_MOTOR_diff_collapsed,ICPS_soc_late_DLPFC_diff_collapsed,ICPS_soc_late_OCC_diff_collapsed,ICPS_soc_late_MOTOR_diff_collapsed,ICPS_nonsoc_early_DLPFC_diff_collapsed,ICPS_nonsoc_early_OCC_diff_collapsed,ICPS_nonsoc_early_MOTOR_diff_collapsed,ICPS_nonsoc_late_DLPFC_diff_collapsed,ICPS_nonsoc_late_OCC_diff_collapsed,ICPS_no

In [ ]:
print(
    (merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_nonsoc_early_DLPFC_R_err"] + \
merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_nonsoc_early_DLPFC_L_err"]
) / 2,

(merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_nonsoc_early_DLPFC_R_corr"] + \
merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_nonsoc_early_DLPFC_L_corr"]
) / 2,

    (merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_soc_early_DLPFC_R_err"] + \
merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_soc_early_DLPFC_L_err"]
) / 2,

    (merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_soc_early_DLPFC_R_corr"] + \
merged_valid_eeg_data[merged_valid_eeg_data["sub"] == 3000026]["ICPS_soc_early_DLPFC_L_corr"]
) / 2,
)

In [ ]:
import pandas as pd

# 1. Define your key columns
keys = ['sub', 'soc', 'acc']

# 2. Prepare the DataFrames
# Ensure both have the same index for comparison
# We also sort the index to ensure alignment matches perfectly

df_old = pd.read_csv(find_newest_file("ms*long*.csv"))
print(df_old.shape)
df_new = pd.read_csv(find_newest_file("thrive*long*.csv")).iloc[:df_old.shape[0], :df_old.shape[1]]
print(df_new.shape)

df_old_indexed = df_old.set_index(keys).sort_index()
df_new_indexed = df_new.set_index(keys).sort_index()

# 3. Align Columns
# Keep only columns that exist in BOTH dataframes to avoid errors
common_cols = df_old_indexed.columns.intersection(df_new_indexed.columns)
df_old_ready = df_old_indexed[common_cols]
df_new_ready = df_new_indexed[common_cols]

# 4. Run the Comparison
# result will show ONLY the values that differ. 
# If a value is the same, it shows NaN (or is hidden depending on options).
diff = df_new_ready.compare(df_old_ready, align_axis=1)

# 5. Display Results
if diff.empty:
    print("✅ The DataFrames are identical for all common columns.")
else:
    print(f"⚠️ Found differences in {len(diff)} rows.")
    
    # Rename columns for clarity (self=new, other=old)
    diff = diff.rename(columns={'self': 'new', 'other': 'old'}, level=1)
    
    # Show the first few differences
    print(diff.head())

    # Optional: Save to CSV to inspect all mismatches
    # diff.to_csv("comparison_diff.csv")

In [ ]:
import pandas as pd

# 1. Setup
keys = ['sub', 'soc', 'acc']
df_old = pd.read_csv(find_newest_file("ms*long*.csv"))
print(df_old.shape)
df_new = pd.read_csv(find_newest_file("thrive*long*.csv")).iloc[:df_old.shape[0], :df_old.shape[1]]
print(df_new.shape)
common_cols = df_old_indexed.columns.intersection(df_new_indexed.columns)

# 2. Run Comparison with keep_equal=True
# keep_equal=True -> Shows the actual values instead of NaN when they match
diff = df_new_indexed[common_cols].compare(df_old_indexed[common_cols], 
                                           align_axis=1, 
                                           keep_equal=True)

if diff.empty:
    print("✅ Identical.")
else:
    print(f"⚠️ Found differences in {len(diff)} rows.")
    
    # Rename for clarity
    diff = diff.rename(columns={'self': 'new', 'other': 'old'}, level=1)
    
    # --- Option A: The Table View (Now with real numbers, no NaNs) ---
    print("\n--- Wide View (Context included) ---")
    print(diff.head())

    # --- Option B: The "Just the Differences" List ---
    # This transforms the wide table into a simple list of WHAT changed
    print("\n--- Stacked View (Only changes) ---")
    
    # We run compare again strictly to find differences (default behavior)
    diff_strict = df_new_indexed[common_cols].compare(df_old_indexed[common_cols])
    
    # Stack it to get a list: Sub | Column | New Value | Old Value
    diff_stack = diff_strict.stack(level=0)
    diff_stack = diff_stack.reset_index().rename(columns={'level_3': 'column_name'})
    
    print(diff_stack.head(10))

In [ ]:
# 1. Compare WITHOUT keeping equal values (Matches become NaNs)
diff = df_new_indexed.compare(df_old_indexed)

# 2. Stack the result to get a clean list
# This drops all the NaNs (matches), leaving only real differences
diff_list = diff.stack(level=0)
diff_list = diff_list.reset_index().rename(columns={'level_3': 'variable'})

# 3. View the clean list
print(f"⚠️ Found {len(diff_list)} specific mismatches.")
print(diff_list)

In [ ]:
import numpy as np

def analyze_mismatches(df_old, df_new, key="sub"):
    """
    Compares two dataframes row-by-row and prints mismatches.
    """
    # 1. Align Dataframes on Subject ID to ensure we compare the same people
    # Suffixes help us distinguish old vs new values
    merged = df_old.merge(df_new, on=key, suffixes=('_old', '_new'), how='inner')
    
    # Identify columns to check (remove 'sub' and keep only shared columns)
    common_cols = [c for c in df_old.columns if c in df_new.columns and c != key]
    
    print(f"--- Mismatch Report ---")
    
    for col in common_cols:
        col_old = f"{col}_old"
        col_new = f"{col}_new"
        
        # 2. Robust Comparison Logic
        # We use a mask that accounts for two common false alarms:
        # A) NaN != NaN (Python says this is True, we want False)
        # B) 1.0000001 != 1.0 (Floating point math differences)
        
        # Check if values are NOT close (for numbers) AND NOT both NaN
        # If strings/mixed, simple inequality is used.
        if np.issubdtype(merged[col_old].dtype, np.number) and np.issubdtype(merged[col_new].dtype, np.number):
            # Check for numeric difference beyond a tiny tolerance (1e-9)
            # & checks where BOTH are not NaN (if one is NaN and other is 5, it's a mismatch)
            
            # Are they exactly NaN in the same places?
            nan_mismatch = merged[col_old].isna() != merged[col_new].isna()
            
            # Are the numbers different? (Ignored where NaNs exist)
            numeric_mismatch = ~np.isclose(merged[col_old], merged[col_new], equal_nan=True)
            
            is_diff = nan_mismatch | numeric_mismatch
        else:
            # Simple inequality for non-numbers
            is_diff = merged[col_old] != merged[col_new]
            
        # 3. Filter the rows that differ
        bad_rows = merged.loc[is_diff, [key, col_old, col_new]]
        
        if not bad_rows.empty:
            print(f"\nCOLUMN: {col} | {len(bad_rows)} mismatches found")
            print(bad_rows.head(5)) # Print first 5 errors to avoid flooding console
            print("...")

# Run the checker
old_df = pd.read_csv(find_newest_file("ms*long*.csv"))
new_df = pd.read_csv(find_newest_file("thrive*long*.csv"))#.iloc[:-166, :]

analyze_mismatches(old_df, new_df)

In [ ]:
new_df["ERN_min_CRN"]

In [ ]:
old_df["ERN_min_CRN"]

In [ ]:
new_df

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd
from glob import glob
import datetime
import time
import re
import h5py
import os
import sys

# session = sys.argv[1]
dataset_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"

arr_path = f"{dataset_path}/derivatives/preprocessed/TF_arrays/{session}/old/"
helper_data = h5py.File(
    glob(f"{dataset_path}/derivatives/preprocessed/TF_outputs/{session}/resp/seed_1/TF/sub-*.mat")[0]
)

freqs = helper_data['frequency'][:]
times = helper_data['ds_time'][:]
assert np.max(np.abs(times)) > 50, (
    f"Time unit warning: Max time is {np.max(np.abs(times)):.2f}. "
    "This looks like SECONDS. Convert 'times' to MS (times * 1000)."
)
ch_locs = [str(i) for i in range(1, 65)]

# NOT DIFFERENCE ICPS

matching_files = glob(f"{dataset_path}/derivatives/behavior/{session}/*summary*{session}*.csv")

# Check if any files were found
if not matching_files:
    print("No matching files found.")
else:
    # Find the newest file based on modification time
    new_file_path = max(matching_files, key=os.path.getmtime)
    thrive_data = pd.read_csv(new_file_path)["sub"].to_frame()
    print(f"The newest file is: {new_file_path}")

measures = [
    "ICPS",
]
conditions = [
    "resp_s_i_0",
    "resp_s_i_1",
    # "resp_s_c_1",
    "resp_ns_i_0",
    "resp_ns_i_1",
    # "resp_ns_c_1",
            ]

for m in measures:
    for c in conditions:
        tf_arr = scipy.io.loadmat(f"{arr_path}/{m}_{c}.mat")
        tf_data = tf_arr[f"{m}_{c}"]       
        assert tf_data.shape[1:] == (64, 375, 59), f"Check your {m} data!"
        sub_ids = tf_arr['subjects']
        pattern = r'sub-(\d+)'
        sub_ids = [int(re.search(pattern, i).group(1)) for i in sub_ids]

        for band in [
            "theta",
             # "delta"
                    ]:
            for window in [
                "early",
                # "late"
            ]:
                for cluster in [
                    "DLPFC_L",
                    "DLPFC_R",
                    # "OCC_L",
                    # "OCC_R",
                    # "MOTOR_L",
                    # "MOTOR_R",
                    #"CENTRAL"
                ]:
                    ch = None

                    if cluster == "DLPFC_L":
                        ch = ['6', '9']
                    elif cluster == "DLPFC_R":
                        ch = ['39', '42']
                    elif cluster == "OCC_L":
                        ch = ['22', '24']
                    elif cluster == "OCC_R":
                        ch = ['53', '55']
                    elif cluster == "MOTOR_L":
                        ch = ['3', '7']
                    elif cluster == "MOTOR_R":
                        ch = ['35', '40']
                    elif cluster == "CENTRAL":
                        ch = ['19', '50']

                    assert ch is not None, f"Cluster {cluster} is not defined in channel map!"

                    if band == "theta":
                        fmin = 4
                        fmax = 7
                    elif band == "delta":
                        fmin = 1
                        fmax = 3
    
                    if window == "early":
                        tmin = 0
                        tmax = 250
                    elif window == "late":
                        tmin = 256
                        tmax = 504
                        
                    fmin_idx = np.argmin(np.abs(freqs-fmin))
                    assert freqs[fmin_idx] == fmin, "Check your freqs!"
                    fmax_idx = np.argmin(np.abs(freqs-fmax))
                    assert freqs[fmax_idx] == fmax, "Check your freqs!"
                    
                    tmin_idx = np.argmin(np.abs(times-tmin))
                    # assert times[tmin_idx] == tmin, "Check your times!"
                    tmax_idx = np.argmin(np.abs(times-tmax))
                    # assert times[tmax_idx] == tmax, "Check your times!"
                    
                    ch_idx = []
                    for channel in ch:
                        if channel in ch_locs:
                            ch_idx.append(ch_locs.index(channel))
                    
                    # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
                    tf_df = pd.DataFrame(columns = ["sub", f"{m}_{c}_{band}_{window}_{cluster}"])
                    
                    # for sub_id in sub_idx:
                    for sub_id in range(tf_data.shape[0]):
                        # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
                        sub_avg = tf_data[sub_id, :, :, :]
                        assert sub_avg.shape == (64, 375, 59), f"Check your {m} data!"
                        
                        ch_avg = np.mean(sub_avg[ch_idx, :, :], 0)
                        assert ch_avg.shape == (375, 59), f"Check your {m} data!"
                        
                        time_avg = np.mean(ch_avg[tmin_idx:tmax_idx+1, :], 0)
                        assert len(time_avg) == 59 and time_avg.ndim == 1, f"Check your {m} data!"
                        freq_avg = np.mean(time_avg[fmin_idx:fmax_idx+1], 0)
                    
                        tf_df.loc[sub_id, "sub"] = sub_ids[sub_id]
                        tf_df.loc[sub_id, f"{m}_{c}_{band}_{window}_{cluster}"] = freq_avg
                    
                    thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

thrive_data = thrive_data[
[i for i in thrive_data.columns if ("delta" not in i or i == "sub")]
]

# note that this renaming logic would not work correctly if congruent conditions are requested above
colnames = list(thrive_data.columns)
for i, c in enumerate(colnames[1:]):
    i+=1
    splitted_list = c.split("_")
    if splitted_list[2] == "s":
        splitted_list[2] = "soc"
    elif splitted_list[2] == "ns":
        splitted_list[2] = "nonsoc"
    if splitted_list[4] == "0":
        splitted_list[4] = "err"
    elif splitted_list[4] == "1":
        splitted_list[4] = "corr"
    splitted_list[1] = ""
    splitted_list[5] = ""
    splitted_list[3] = ""
    splitted_list = [i for i in splitted_list if i!=""]
    colnames[i] = "_".join(splitted_list)

thrive_data.columns = colnames

# thrive_data.to_csv(f"{dataset_path}/derivatives/csv/{session}/thrive_icps_{datetime.datetime.now()}.csv", index=False)

In [ ]:
old_df = pd.read_csv("tf_data_collapsed_merged.csv")
# old_df = old_df[old_df["hemisphere"] == "L"]
list_of_diff = []
old_col = "ICPS_DLPFC"
new_col = "ICPS_early_DLPFC_collapsed"
for sub in old_df["sub"].unique():
    for soc in [0, 1]:
        for acc in [0, 1]:
            if len(df_long[(df_long["sub"] == sub) & (df_long["soc"] == soc) & (df_long["acc"] == acc)][new_col]) > 0:
                res = old_df[(old_df["sub"] == sub) & (old_df["soc"] == soc) & (old_df["acc"] == acc)][old_col].item() == \
                df_long[(df_long["sub"] == sub) & (df_long["soc"] == soc) & (df_long["acc"] == acc)][new_col].item()
                if res == False:
                    # print(sub)
                    list_of_diff.append(sub)

## Trash

In [ ]:
def zscore_dataframe(df, exclude_cols=None):
    """
    Returns a z-scored copy of the dataframe, ignoring specified columns.
    """
    if exclude_cols is None:
        exclude_cols = []
    
    df_out = df.copy()
    
    # Identify columns to normalize
    # Optional: Add strict check for numeric types if mixing data types
    cols_to_norm = [c for c in df.columns if c not in exclude_cols]
    
    # Apply Z-score: (Value - Mean) / Std Dev
    # Note: Pandas uses ddof=1 (sample standard deviation) by default
    df_out[cols_to_norm] = (
        df_out[cols_to_norm] - df_out[cols_to_norm].mean()
    ) / df_out[cols_to_norm].std()
    
    return df_out

In [ ]:
# for col in ddm_df:
#     if col == "sub" or col == "soc" or col == "pre_accuracy" or col == "condition":
#         continue
#     else:
#         print(
#             list(df_pivoted[f"{col}_postcorr_soc"]) == list(ddm_df_soc[ddm_df_soc["pre_accuracy"] == 1][f"{col}"]),
#             list(df_pivoted[f"{col}_posterr_soc"]) == list(ddm_df_soc[ddm_df_soc["pre_accuracy"] == 0][f"{col}"]),
#             list(df_pivoted[f"{col}_postcorr_nonsoc"]) == list(ddm_df_nonsoc[ddm_df_nonsoc["pre_accuracy"] == 1][f"{col}"]),
#             list(df_pivoted[f"{col}_posterr_nonsoc"]) == list(ddm_df_nonsoc[ddm_df_nonsoc["pre_accuracy"] == 0][f"{col}"]),
#         )

In [ ]:
# pd.read_csv(glob(f"{analysis_path}/derivatives/csv/{session}/thrive_erp.csv")[0])
# pd.read_csv(glob(f"{analysis_path}/derivatives/csv/{session}/thrive_erp_laplacian.csv")[0])

In [ ]:
#merged_valid_eeg_data = merged_valid_eeg_data.merge(new_state_survey_df, on="sub", how="left")
merged_valid_eeg_data = merged_valid_eeg_data.merge(state_surveys, on="sub", how="left")

In [ ]:
session = "s1_r1"
merged_valid_eeg_data = pd.read_csv("thrive_mega_df_2025-03-25_21-11-17.csv")
merged_valid_eeg_data.columns = [(i + "_" + session) if (session not in i and i!="sub") else i for i in merged_valid_eeg_data.columns]
merged_valid_eeg_data = merged_valid_eeg_data.merge(pd.read_csv(
    "/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/csv/s2_r1/thrive_mega_df_s2_r1_2025-05-07_19-49-50.csv"), on="sub", how="left")

list(merged_valid_eeg_data.columns)

In [ ]:
merged_valid_eeg_data.columns = [(i + "_" + session) if (session not in i and i!="sub") else i for i in merged_valid_eeg_data.columns]

In [ ]:
date_now = datetime.today().strftime('%Y-%m-%d_%H-%M-%S')

# merged_valid_eeg_data.to_csv(f"{analysis_path}/derivatives/csv/{session}/thrive_mega_df_{session}_{date_now}.csv", index=False)
merged_valid_eeg_data.to_csv(f"{analysis_path}/derivatives/csv/s1_r1/thrive_mega_df_s1_s2_{date_now}.csv", index=False)

# Merge behav and FNE for LDA

In [ ]:
import os
import pandas as pd
from glob import glob
import numpy as np
from functools import reduce
from datetime import datetime
import re

In [ ]:
def replace_outliers_with_nan(df, sd_thresh = 3):
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            mean = df[column].mean()
            std = df[column].std()
            threshold_upper = mean + sd_thresh * std
            threshold_lower = mean - sd_thresh * std
            
            # Replace outliers with NaN
            df[column] = df[column].apply(lambda x: np.nan if (x > threshold_upper or x < threshold_lower) else x)
    return df

def replace_outliers_with_nan_cols(df, columns_to_check, sd_thresh=3):
    for column in columns_to_check:
        if column in df.columns and pd.api.types.is_numeric_dtype(df[column]):
            mean = df[column].mean()
            std = df[column].std()
            threshold_upper = mean + sd_thresh * std
            threshold_lower = mean - sd_thresh * std

            # Replace outliers with NaN
            df[column] = df[column].apply(lambda x: np.nan if (x > threshold_upper or x < threshold_lower) else x)
    return df

In [ ]:


# merged_valid_eeg_data = merged_valid_eeg_data.merge(behav_data, on="sub", how="left")

In [ ]:
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"
dfs = []
for session in ["s1_r1", "s2_r1", "s3_r1"]:
    # list_of_data = []
    print(session)
    pattern = r'(\d{4}-\d{2}-\d{2}_\d{4})'
    most_recent_iqs = max(
        glob(f"{dataset_path}/derivatives/preprocessed/redcap/Thrive*iqschild{"".join(session.split("_"))}*.csv"),
        key=lambda x: re.search(pattern, x).group(1)
    )
    print(most_recent_iqs)
    iqs_ch = pd.read_csv(most_recent_iqs)
    iqs_ch = iqs_ch.rename({"record_id": "sub"}, axis=1)
    iqs_ch = iqs_ch[["sub", f"bfne_b_scrdTotal_{session}_e1"]]
    # print(iqs_ch[f"bfne_b_scrdTotal_{session}_e1"].dropna().shape[0])
    # list_of_data.append(iqs_ch)

    if session == "s1_r1":
        iqs_parent_path = max(
            glob(f"{dataset_path}/sourcedata/checked/redcap/Thrive*iqsparent{"".join(session.split("_"))}*.csv"),
            key=lambda x: re.search(pattern, x).group(1)
        )
        print(iqs_parent_path)
        age_data = pd.read_csv(iqs_parent_path)
        age_data = age_data[[i for i in age_data.columns if "agemos" in i or "record_id" in i]]
        age_data = age_data.rename({"record_id" : "sub"}, axis=1)
        age_data["sub"] = age_data["sub"] - 80000
        
        age_data = age_data.dropna(how="all", subset = age_data.columns[1:]).reset_index(drop=True)
        
        # to merge Eng and Spanish versions
        for i in range(age_data.shape[0]):
            age_data.loc[i, "age_m"] = age_data.iloc[i, 1:].dropna().values[0]
        
        age_data = age_data[["sub", "age_m"]]
        age_data = age_data.rename({"age_m": "age"}, axis=1)
    
    # if session == "s2_r1":
    #     age_data["age"] = age_data["age"] + 9
    # elif session == "s3_r1":
    #     age_data["age"] = age_data["age"] + 18
    
        age_data.columns = [i+f"_{session}_e1" if i != "sub" else i for i in age_data.columns]
        # list_of_data.append(age_data)

    # redcap_data = iqs_ch.merge(age_data, on="sub", how="left")
    
    pattern = r'(\d{2}_\d{2}_\d{4}_\d{2}_\d{2}_\d{2})'
    behavivor_summary_path = max(
        glob(f"{analysis_path}/derivatives/behavior/{session}/summary*{session}*.csv"),
        key=lambda x: re.search(pattern, x).group(1)
    )
    
    behavior_df = pd.read_csv(behavivor_summary_path)
    
    behavior_df_soc = behavior_df[[col for col in behavior_df.columns if ("_soc" in col or "sub" in col)]]
    
    behavior_df_soc = behavior_df_soc[behavior_df_soc["acc_soc"] >= 0.6]
    behavior_df_soc = behavior_df_soc[behavior_df_soc["6_or_more_err_soc"] == 1]
    behavior_df_soc = replace_outliers_with_nan_cols(behavior_df_soc, ["invalid_rt_percent_soc", "skipped_percent_soc"])
    behavior_df_soc = behavior_df_soc.dropna(subset="invalid_rt_percent_soc")
    behavior_df_soc = behavior_df_soc.dropna(subset="skipped_percent_soc")
    
    behav_data = behavior_df_soc[["sub", "peri_rt_soc"]]
    behav_data.columns = [i+f"_{session}_e1" if i != "sub" else i for i in behav_data.columns]
    
    full_data = behav_data.merge(iqs_ch, on="sub", how="left")
    if session == "s1_r1":
        full_data = full_data.merge(age_data, on="sub", how="left")
    # list_of_data.append(behav_data)
    # full_data = reduce(lambda left, right: pd.merge(left, right, on='sub', how='left'), list_of_data)
    # full_data = replace_outliers_with_nan(full_data)
        
    dfs.append(full_data)

merged_df = reduce(lambda left, right: pd.merge(left, right, on='sub', how='left'), dfs)
for session in ["s2_r1", "s3_r1"]:
    if session == "s2_r1":
        merged_df[f"age_{session}_e1"] = merged_df["age_s1_r1_e1"] + 9
    elif session == "s3_r1":
        merged_df[f"age_{session}_e1"] = merged_df["age_s1_r1_e1"] + 18
for age_column in [i for i in merged_df.columns if "age" in i]:
    merged_df[age_column] = merged_df[age_column]/12

merged_df = merged_df.dropna().reset_index(drop=True)
merged_df.columns = [i.replace("_r1_e1", "") if i != "sub" else i for i in merged_df.columns]
merged_df.columns = [i.replace("_b_scrdTotal", "") if "bfne" in i else i for i in merged_df.columns]
merged_df.columns = [i.replace("_rt_soc", "") if "peri" in i else i for i in merged_df.columns]

merged_df

In [ ]:
merged_df.to_csv("LDA_data.csv", index=False)

In [ ]:
dfs[0][dfs[0]["sub"] == 3000123]

In [ ]:
behav_data.merge(iqs_ch[["sub", f"bfne_b_scrdTotal_{session}_e1"]].dropna(), on="sub", how="left").dropna()